In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import col, monotonically_increasing_id, regexp_replace
import IPython.display as ipyd
from pyspark.sql.functions import col, regexp_replace, monotonically_increasing_id, when
from pyspark.sql.functions import col, regexp_replace, to_date


In [2]:
spark = SparkSession.builder.getOrCreate()
spark.conf.set("spark.sql.files.ignoreCorruptFiles", "true")
df = spark.read.parquet("/home/iceberg/data/viagens") #2022,2023, 2024 e 2025 estao corrupted
df = df.withColumn("id", monotonically_increasing_id())

df_typecasted = df \
    .withColumn("ano", col("ano").cast("int")) \
    .withColumn("mes", col("mes").cast("int")) \
    .withColumn("Valor passagens", regexp_replace(col("Valor passagens"), ",", ".").cast("float")) \
    .withColumn("Valor diárias", regexp_replace(col("Valor diárias"), ",", ".").cast("float")) \
    .withColumn("Valor devolução", regexp_replace(col("Valor devolução"), ",", ".").cast("float")) \
    .withColumn("Valor outros gastos", regexp_replace(col("Valor outros gastos"), ",", ".").cast("float")) \
    .withColumn("Período - Data de início", to_date(col("Período - Data de início"), "dd/MM/yyyy")) \
    .withColumn("Período - Data de fim", to_date(col("Período - Data de fim"), "dd/MM/yyyy")) \
    .withColumn(
        "Viagem Urgente", when(col("Viagem Urgente") == "SIM", True)
        .when(col("Viagem Urgente") == "NÃO", False)
        .otherwise(None)
)
cardinality = sorted([
    (column, df_typecasted.select(F.approx_count_distinct(column)).collect()[0][0])
    for column in df_typecasted.columns if column != "id"
], key=lambda x: x[1])

ordered_cols = [c for c, _ in cardinality]

df_final = df_typecasted.select(ordered_cols) \
                        .orderBy(*[c for c, _ in cardinality])

26/04/16 23:29:49 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/04/16 23:29:49 WARN FileScanRDD: Skipped the rest of the content in the corrupted file: path: file:///home/iceberg/data/viagens/ano=2023/mes=unknown/tmpt8xmhz48.parquet, range: 0-121093695, partition values: [2023,unknown]
java.io.IOException: Expected 54875 values in column chunk at file:/home/iceberg/data/viagens/ano=2023/mes=unknown/tmpt8xmhz48.parquet offset 227491 but got 56305 values instead over 4 pages ending at file offset 361520
	at org.apache.parquet.hadoop.ParquetFileReader$Chunk.readAllPages(ParquetFileReader.java:1662)
	at org.apache.parquet.hadoop.ParquetFileReader$Chunk.readAllPages(ParquetFileReader.java:1547)
	at org.apache.parquet.hadoop.ParquetFileReader.readChunkPages(ParquetFileReader.java:1157)
	at org.apache.parquet.hadoop.ParquetFileReader.internalReadRowGroup(ParquetFileR

In [34]:
spark.sql("CREATE NAMESPACE IF NOT EXISTS demo.viagens")

df_final.writeTo("demo.compras.viagens") \
        .tableProperty("format-version", "2") \
        .partitionedBy("ano", "mes") \
        .createOrReplace()

[Stage 172:=========================>                              (6 + 7) / 13]

[3656.122s][warning][gc,alloc] Executor task launch worker for task 0.0 in stage 172.0 (TID 1314): Retried waiting for GCLocker too often allocating 131074 words


26/04/16 22:54:11 ERROR Executor: Exception in task 0.0 in stage 172.0 (TID 1314)
java.lang.OutOfMemoryError: Java heap space
	at org.apache.spark.util.collection.unsafe.sort.UnsafeSorterSpillReader.<init>(UnsafeSorterSpillReader.java:53)
	at org.apache.spark.util.collection.unsafe.sort.UnsafeSorterSpillWriter.getReader(UnsafeSorterSpillWriter.java:159)
	at org.apache.spark.util.collection.unsafe.sort.UnsafeExternalSorter.getSortedIterator(UnsafeExternalSorter.java:555)
	at org.apache.spark.sql.execution.UnsafeExternalRowSorter.sort(UnsafeExternalRowSorter.java:172)
	at org.apache.spark.sql.catalyst.expressions.GeneratedClass$GeneratedIteratorForCodegenStage2.processNext(Unknown Source)
	at org.apache.spark.sql.execution.BufferedRowIterator.hasNext(BufferedRowIterator.java:43)
	at org.apache.spark.sql.execution.WholeStageCodegenEvaluatorFactory$WholeStageCodegenPartitionEvaluator$$anon$1.hasNext(WholeStageCodegenEvaluatorFactory.scala:43)
	at scala.collection.Iterator$$anon$10.hasNext(

ConnectionRefusedError: [Errno 111] Connection refused

Traceback (most recent call last):
  File "/usr/local/lib/python3.10/runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/usr/local/lib/python3.10/runpy.py", line 86, in _run_code
    exec(code, run_globals)
  File "/usr/local/lib/python3.10/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/usr/local/lib/python3.10/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  File "/usr/local/lib/python3.10/site-packages/ipykernel/kernelapp.py", line 739, in start
    self.io_loop.start()
  File "/usr/local/lib/python3.10/site-packages/tornado/platform/asyncio.py", line 205, in start
    self.asyncio_loop.run_forever()
  File "/usr/local/lib/python3.10/asyncio/base_events.py", line 603, in run_forever
    self._run_once()
  File "/usr/local/lib/python3.10/asyncio/base_events.py", line 1871, in _run_once
    event_list = self._selector.select(timeout)
  File "